In [1]:
library(ggplot2)
library(dplyr)
library(Seurat)
library(tidyverse)
library(tidyr)
library(stringr)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.3.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘tidyverse’ was built under R version 4.3.3”
Warning message:
“package ‘readr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.2”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
── Attaching core tidyverse packages ────────────────────────────────

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
# indir=""
# outdir=""

In [ ]:
specific_hyper_hypo_DHMR <- read.csv(paste0(indir,"/00-dhmr_specific_hyper_hypo_DHMR_include_region.csv"),row.names=1,check.names=F)

In [3]:
hyper_regions <- specific_hyper_hypo_DHMR %>%
  select(Subclass, Specific_Hyper_Region) %>%
  # Replace single quotes with double quotes
  mutate(Specific_Hyper_Region = str_replace_all(Specific_Hyper_Region, "'", "\"")) %>%
  # Converts a string to a real list
  mutate(Specific_Hyper_Region = map(Specific_Hyper_Region, ~ jsonlite::fromJSON(.))) %>%
  # Expand into separate rows
  unnest(Specific_Hyper_Region) %>%
  rename(region = Specific_Hyper_Region) %>% data.frame()

In [4]:
head(hyper_regions);dim(hyper_regions)

,Subclass,region
,<chr>,<chr>
1,L2/3 IT CTX Glut,chr1_40205592_40205913
2,L2/3 IT CTX Glut,chr1_63801218_63802655
3,L2/3 IT CTX Glut,chr1_95910842_95911315
4,L2/3 IT CTX Glut,chr1_121992803_121993033
5,L2/3 IT CTX Glut,chr1_123859818_123861293
6,L2/3 IT CTX Glut,chr1_127035200_127035964


[1] 122947      2

In [5]:
unique(hyper_regions$Subclass);length(unique(hyper_regions$Subclass))

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

[1] 25

In [ ]:
process_regions <- function(subclass1,subclass2) { # 'L2/3 IT CTX Glut', 'L2_3'
  df <- hyper_regions[hyper_regions$Subclass == subclass1,]
  df_split <- do.call(rbind, strsplit(df$region, "_"))
  df_split <- as.data.frame(df_split, stringsAsFactors = FALSE)
  colnames(df_split) <- c("chr", "start", "end")
  df_split$start <- as.numeric(df_split$start)
  df_split$end <- as.numeric(df_split$end)
  df_split$mid <- (df_split$start + df_split$end) / 2
  df_split$new_start <- pmax(0, df_split$mid - 3000)
  df_split$new_end <- df_split$mid + 3000

  df_final <- data.frame(
    chr = df_split$chr,
    start = as.integer(df_split$new_start),
    end = as.integer(df_split$new_end),
    region = paste0(df_split$chr, ":", as.integer(df_split$new_start), "_", as.integer(df_split$new_end))
  )
  write.table(df_final,
              sprintf("%s/02_DHMR_hyper_bed/%s_hyper_DHMR.bed",outdir,subclass2),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
  print(head(df_final))
  print(dim(df_final))
}

In [7]:
process_regions('L2/3 IT CTX Glut', 'L2_3')

   chr     start       end                   region
1 chr1  40202752  40208752   chr1:40202752_40208752
2 chr1  63798936  63804936   chr1:63798936_63804936
3 chr1  95908078  95914078   chr1:95908078_95914078
4 chr1 121989918 121995918 chr1:121989918_121995918
5 chr1 123857555 123863555 chr1:123857555_123863555
6 chr1 127032582 127038582 chr1:127032582_127038582
[1] 157   4


In [8]:
process_regions('L4/5 IT CTX Glut', 'L4_5')

   chr    start      end                 region
1 chr1  3987106  3993106   chr1:3987106_3993106
2 chr1  7534017  7540017   chr1:7534017_7540017
3 chr1  9778315  9784315   chr1:9778315_9784315
4 chr1 14836573 14842573 chr1:14836573_14842573
5 chr1 14912279 14918279 chr1:14912279_14918279
6 chr1 17753923 17759923 chr1:17753923_17759923
[1] 1392    4


In [9]:
process_regions('L5 IT CTX Glut', 'L5')

   chr    start      end                 region
1 chr1  4183172  4189172   chr1:4183172_4189172
2 chr1  9992050  9998050   chr1:9992050_9998050
3 chr1 20080523 20086523 chr1:20080523_20086523
4 chr1 22217957 22223957 chr1:22217957_22223957
5 chr1 25589289 25595289 chr1:25589289_25595289
6 chr1 26493784 26499784 chr1:26493784_26499784
[1] 632   4


In [10]:
process_regions('L5 NP CTX Glut', 'NP')

   chr   start     end               region
1 chr1 4033189 4039189 chr1:4033189_4039189
2 chr1 4930811 4936811 chr1:4930811_4936811
3 chr1 6075304 6081304 chr1:6075304_6081304
4 chr1 6364959 6370959 chr1:6364959_6370959
5 chr1 7731892 7737892 chr1:7731892_7737892
6 chr1 9364957 9370957 chr1:9364957_9370957
[1] 9693    4


In [11]:
process_regions('L6 CT CTX Glut', 'CT')

   chr    start      end                 region
1 chr1  3905893  3911893   chr1:3905893_3911893
2 chr1  5414853  5420853   chr1:5414853_5420853
3 chr1  5955407  5961407   chr1:5955407_5961407
4 chr1  9592469  9598469   chr1:9592469_9598469
5 chr1 11618244 11624244 chr1:11618244_11624244
6 chr1 11725992 11731992 chr1:11725992_11731992
[1] 1243    4


In [12]:
process_regions('DG Glut', 'DG')

   chr    start      end                 region
1 chr1  5010021  5016021   chr1:5010021_5016021
2 chr1  6860106  6866106   chr1:6860106_6866106
3 chr1  7356020  7362020   chr1:7356020_7362020
4 chr1 13908325 13914325 chr1:13908325_13914325
5 chr1 16811867 16817867 chr1:16811867_16817867
6 chr1 17027488 17033488 chr1:17027488_17033488
[1] 687   4


In [13]:
process_regions('CA1-ProS Glut', 'CA1')

   chr   start     end               region
1 chr1 3704740 3710740 chr1:3704740_3710740
2 chr1 3865830 3871830 chr1:3865830_3871830
3 chr1 3998351 4004351 chr1:3998351_4004351
4 chr1 4015053 4021053 chr1:4015053_4021053
5 chr1 4165527 4171527 chr1:4165527_4171527
6 chr1 4247138 4253138 chr1:4247138_4253138
[1] 2566    4


In [14]:
process_regions('CA3 Glut', 'CA23')

   chr   start     end               region
1 chr1 3148805 3154805 chr1:3148805_3154805
2 chr1 3774023 3780023 chr1:3774023_3780023
3 chr1 4028936 4034936 chr1:4028936_4034936
4 chr1 4054509 4060509 chr1:4054509_4060509
5 chr1 4062707 4068707 chr1:4062707_4068707
6 chr1 4069607 4075607 chr1:4069607_4075607
[1] 3974    4


In [15]:
process_regions('Pvalb Gaba', 'Pvalb')

   chr   start     end               region
1 chr1 3166178 3172178 chr1:3166178_3172178
2 chr1 3185767 3191767 chr1:3185767_3191767
3 chr1 3186654 3192654 chr1:3186654_3192654
4 chr1 3227235 3233235 chr1:3227235_3233235
5 chr1 3255328 3261328 chr1:3255328_3261328
6 chr1 3272551 3278551 chr1:3272551_3278551
[1] 13766     4


In [16]:
process_regions('Sst Gaba', 'Sst')

   chr   start     end               region
1 chr1 3071678 3077678 chr1:3071678_3077678
2 chr1 3081489 3087489 chr1:3081489_3087489
3 chr1 3088516 3094516 chr1:3088516_3094516
4 chr1 3094126 3100126 chr1:3094126_3100126
5 chr1 3095825 3101825 chr1:3095825_3101825
6 chr1 3101626 3107626 chr1:3101626_3107626
[1] 17454     4


In [17]:
process_regions('Sncg Gaba', 'Sncg')

   chr   start     end               region
1 chr1 3237126 3243126 chr1:3237126_3243126
2 chr1 3595456 3601456 chr1:3595456_3601456
3 chr1 3843437 3849437 chr1:3843437_3849437
4 chr1 7204224 7210224 chr1:7204224_7210224
5 chr1 7287162 7293162 chr1:7287162_7293162
6 chr1 7544090 7550090 chr1:7544090_7550090
[1] 5165    4


In [18]:
process_regions('Lamp5 Gaba', 'Lamp5')

   chr   start     end               region
1 chr1 3030526 3036526 chr1:3030526_3036526
2 chr1 3114306 3120306 chr1:3114306_3120306
3 chr1 3233121 3239121 chr1:3233121_3239121
4 chr1 3288343 3294343 chr1:3288343_3294343
5 chr1 3507833 3513833 chr1:3507833_3513833
6 chr1 3799917 3805917 chr1:3799917_3805917
[1] 10261     4
